# 03 — Operations & Communication
**Agentic X | Studio 67**

### Overview
This notebook evaluates the **Operations Agent** and tests structured **Agent Communication Language (ACL)** messaging in the simulated Studio 67 event environment. It uses **150 cases and 450 candidate solutions** across Screen Failure, Budget Overrun, and Safety Conflict.

### Agent Roles & Design
- **Operations (Hybrid):** deterministic checks for time, crew, backup equipment, venue access, and supplier availability, followed by a Gemini assessment of practical operational feasibility when all resource checks pass.
- **Communication:** FIPA-style ACL message types **CFP, REQUEST, INFORM, PROPOSE, ACCEPT, REFUSE**, using message IDs, reply linkage, sender/receiver, rounds, and the shared ontology `studio67.event_crisis`.

### Related Python Files
- `agents/operations.py` — operational constraints and Gemini evaluation.
- `core/communication.py` — validated ACL messaging and message history.
- `core/environment.py`, `core/models.py` — dataset, protected agent inputs, shared output schema.
- `agents/finance.py`, `agents/safety.py` — real cross-agent evaluations.

### Requirements & Setup
Python 3.10+, repository `requirements.txt`, and a **GOOGLE_API_KEY** stored in Colab Secrets (or provided securely when prompted). Upload the two updated `.py` files to GitHub **before running**. The notebook imports the source modules and **never overwrites them**.

**Scope:** PEAS and Hybrid agent justification, operational variables/constraints, three realistic scenario tests, all six ACL performatives, communication validation, and integration compatibility. Multi-round agreement, human escalation, and live UI are tested in subsequent notebooks.


### Cell 1 — Project Setup

In [1]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").exists():
    raise RuntimeError("Existing /content/AgenticX is not a Git checkout")
else:
    status = subprocess.run(
        ["git", "-C", str(ROOT), "status", "--porcelain"],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    if status:
        print("Local edits detected: no automatic pull. Check GitHub versions below.")
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)
print("Repository ready:", ROOT)

Repository ready: /content/AgenticX


### Cell 2 — Imports & Source Verification

In [2]:
import importlib
import pandas as pd
from IPython.display import display
from core.environment import load_dataset, get_case, clean, HIDDEN_CASE_FIELDS, HIDDEN_OPTION_FIELDS
from core.models import AgentEvaluation, ACLMessage
import agents.operations as operations_module
import core.communication as communication_module
importlib.reload(operations_module)
importlib.reload(communication_module)
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent

OperationsAgent = operations_module.OperationsAgent
MessageBus = communication_module.MessageBus
assert OperationsAgent.VERSION == "2026-11-operations-v2", "Update agents/operations.py on GitHub and pull latest source"
assert communication_module.ONTOLOGY == "studio67.event_crisis", "Update core/communication.py on GitHub and pull latest source"
assert Path(operations_module.__file__).resolve() == ROOT / "agents/operations.py"
assert Path(communication_module.__file__).resolve() == ROOT / "core/communication.py"
print("Operations and Communication modules verified:", operations_module.__file__, communication_module.__file__, sep="\n")

Operations and Communication modules verified:
/content/AgenticX/agents/operations.py
/content/AgenticX/core/communication.py


### Cell 3 — Dataset & Real Scenario

In [3]:
cases_df, options_df = load_dataset()
assert len(cases_df) == 150 and len(options_df) == 450
assert set(cases_df["scenario"]) == {"Screen Failure", "Budget Overrun", "Safety Conflict"}
case, options = get_case("SCR-001")
assert len(options) == 3
assert all(not (HIDDEN_OPTION_FIELDS & set(o)) for o in options)
assert not (HIDDEN_CASE_FIELDS & set(case))
print("Cases:", len(cases_df), "| Options:", len(options_df))
print("Scenario:", case["scenario"], "| Incident:", case["incident"])
display(pd.DataFrame(options)[["option_id", "option_name", "duration_min", "crew_required", "needs_backup", "supplier_available"]])

Cases: 150 | Options: 450
Scenario: Screen Failure | Incident: Main LED screen failed during final rehearsal


,option_id,option_name,duration_min,crew_required,needs_backup,supplier_available
0,SCR-001-O1,Use backup LED,40,1,True,True
1,SCR-001-O2,Rent replacement screen,70,1,False,True
2,SCR-001-O3,Continue with projection,25,1,False,True


### Cell 4 — Operations PEAS & Agent Type

In [4]:
operations_peas = pd.DataFrame([
    {"Component": "Performance Measure", "Description": "Feasible execution within time, staffing, equipment and access constraints"},
    {"Component": "Environment", "Description": "Studio 67 dynamic event-crisis simulation"},
    {"Component": "Actuators", "Description": "Approve/refuse candidate, assign a score, explain result, communicate via ACL"},
    {"Component": "Sensors", "Description": "Incident, duration, remaining time, crew counts, supplier, backup, venue access"},
])
display(operations_peas)
print("Agent Type: Hybrid (deterministic resource gates + deliberative Gemini operational evaluation).")

,Component,Description
0,Performance Measure,"Feasible execution within time, staffing, equi..."
1,Environment,Studio 67 dynamic event-crisis simulation
2,Actuators,"Approve/refuse candidate, assign a score, expl..."
3,Sensors,"Incident, duration, remaining time, crew count..."


Agent Type: Hybrid (deterministic resource gates + deliberative Gemini operational evaluation).


### Cell 5 — Operational Variables & Parameters

In [5]:
variables = pd.DataFrame([
    ("time_left_min", "Case", "Available minutes", "Reject if duration_min exceeds this value"),
    ("duration_min", "Option", "Required minutes", "Reject when duration exceeds time_left_min"),
    ("crew_count_available", "Case", "Available workers", "Reject when fewer than crew_required"),
    ("crew_available", "Case", "Staff availability flag", "Reject crew-dependent option when False"),
    ("crew_required / needs_crew", "Option", "Staff demand", "Trigger worker checks"),
    ("backup_available / needs_backup", "Case / Option", "Backup supply and demand", "Reject if backup required but unavailable"),
    ("venue_access / requires_venue_access", "Case / Option", "Access and access demand", "Reject if access required but unavailable"),
    ("supplier_available", "Option", "Supplier status", "Reject when not explicitly available"),
    ("AGENTICX_MODEL", "Environment", "Gemini model ID", "Controls provider model used after passing hard gates"),
], columns=["Variable", "Source", "Purpose", "Effect on Agent Decision"])
display(variables)

,Variable,Source,Purpose,Effect on Agent Decision
0,time_left_min,Case,Available minutes,Reject if duration_min exceeds this value
1,duration_min,Option,Required minutes,Reject when duration exceeds time_left_min
2,crew_count_available,Case,Available workers,Reject when fewer than crew_required
3,crew_available,Case,Staff availability flag,Reject crew-dependent option when False
4,crew_required / needs_crew,Option,Staff demand,Trigger worker checks
5,backup_available / needs_backup,Case / Option,Backup supply and demand,Reject if backup required but unavailable
6,venue_access / requires_venue_access,Case / Option,Access and access demand,Reject if access required but unavailable
7,supplier_available,Option,Supplier status,Reject when not explicitly available
8,AGENTICX_MODEL,Environment,Gemini model ID,Controls provider model used after passing har...


### Cell 6 — Validate Operational Constraints

In [6]:
base_case, base_options = get_case("SCR-001")
base_option = base_options[0]
assert OperationsAgent.check_constraints(base_case, base_option)[0], "Expected SCR-001-O1 to be operationally feasible"

constraint_tests = [
    ("Missing time", {**base_case, "time_left_min": None}, base_option, "time"),
    ("Negative time", {**base_case, "time_left_min": -5}, base_option, "time"),
    ("Insufficient time", {**base_case, "time_left_min": 0}, base_option, "time"),
    ("Missing duration", base_case, {**base_option, "duration_min": None}, "time"),
    ("Insufficient crew", {**base_case, "crew_count_available": 0}, base_option, "crew"),
    ("Crew flag unavailable", {**base_case, "crew_available": False}, base_option, "crew"),
    ("Backup unavailable", {**base_case, "backup_available": False}, base_option, "backup"),
    ("Venue closed", {**base_case, "venue_access": False}, base_option, "venue_access"),
    ("Supplier unavailable", base_case, {**base_option, "supplier_available": False}, "supplier"),
    ("Supplier unknown", base_case, {**base_option, "supplier_available": None}, "supplier"),
]
rows = []
for name, test_case, test_option, failed_key in constraint_tests:
    passed, failed, measurements = OperationsAgent.check_constraints(test_case, test_option)
    assert not passed and failed_key in failed
    rows.append({"Test": name, "Passed": passed, "Failed Checks": ", ".join(failed)})
display(pd.DataFrame(rows))

# For hard failures, an actual LLM call must never occur.
class NeverInvoke:
    def invoke(self, prompt):
        raise AssertionError("LLM must not run when operational constraints fail")

gate_agent = OperationsAgent(llm=NeverInvoke())
for name, test_case, test_option, failed_key in constraint_tests:
    result = gate_agent.evaluate(test_case, test_option)
    assert isinstance(result, AgentEvaluation) and not result.approved
    assert result.blocking and result.score == 0 and failed_key in result.reason
print("Operational gate and zero-LLM-call tests passed:", len(constraint_tests))

,Test,Passed,Failed Checks
0,Missing time,False,time
1,Negative time,False,time
2,Insufficient time,False,time
3,Missing duration,False,time
4,Insufficient crew,False,crew
5,Crew flag unavailable,False,crew
6,Backup unavailable,False,backup
7,Venue closed,False,venue_access
8,Supplier unavailable,False,supplier
9,Supplier unknown,False,supplier


Operational gate and zero-LLM-call tests passed: 10


### Cell 7 — Test All Three Scenarios

In [7]:
scenario_rows = []
for case_id in ("SCR-001", "BDG-001", "SFT-001"):
    scenario_case, scenario_options = get_case(case_id)
    selected = next(o for o in scenario_options if float(o["duration_min"]) > 0)
    failed_case = {**scenario_case, "time_left_min": 0}
    result = gate_agent.evaluate(failed_case, selected)
    assert not result.approved and "time" in result.reason
    scenario_rows.append({
        "Case": case_id, "Scenario": scenario_case["scenario"],
        "Option": selected["option_id"], "Available Minutes": 0,
        "Approved": result.approved, "Reason": result.reason,
    })
display(pd.DataFrame(scenario_rows))
print("All three scenario deadline tests passed")

,Case,Scenario,Option,Available Minutes,Approved,Reason
0,SCR-001,Screen Failure,SCR-001-O1,0,False,Operational constraints failed: time. Duration...
1,BDG-001,Budget Overrun,BDG-001-O1,0,False,Operational constraints failed: time. Duration...
2,SFT-001,Safety Conflict,SFT-001-O1,0,False,Operational constraints failed: time. Duration...


All three scenario deadline tests passed


### Cell 8 — Evaluate 450 Options Without LLM Calls

In [8]:
rows = []
for case_row in cases_df.to_dict("records"):
    scenario_case = {k: clean(v) for k, v in case_row.items() if k not in HIDDEN_CASE_FIELDS}
    for opt_row in options_df.loc[options_df["case_id"] == scenario_case["case_id"]].to_dict("records"):
        opt = {k: clean(v) for k, v in opt_row.items() if k not in HIDDEN_OPTION_FIELDS}
        passed, failures, _ = OperationsAgent.check_constraints(scenario_case, opt)
        rows.append({"case_id": scenario_case["case_id"], "scenario": scenario_case["scenario"],
                     "option_id": opt["option_id"], "resource_gates_passed": passed,
                     "failed_checks": ", ".join(failures)})
resource_df = pd.DataFrame(rows)
assert len(resource_df) == 450 and resource_df["case_id"].nunique() == 150
assert resource_df["option_id"].is_unique
display(resource_df.groupby("scenario")["resource_gates_passed"].agg(["count", "sum"]).reset_index())
print("Resource-gate evaluation finished (NOT 450 Gemini evaluations).")

,scenario,count,sum
0,Budget Overrun,150,126
1,Safety Conflict,150,118
2,Screen Failure,150,95


Resource-gate evaluation finished (NOT 450 Gemini evaluations).


### Cell 9 — Gemini Key & Real Operations Agent
**Note — Gemini API Configuration**

The Operations Agent uses Google Gemini for real-time evaluation. For security reasons, the API key is stored in Google Colab Secrets under `GOOGLE_API_KEY` and is not included in the notebook or GitHub repository.

To run the agent, add your own Gemini API key to Colab Secrets and enable notebook access.

In [11]:
import getpass
try:
    from google.colab import userdata
    secret = userdata.get("GOOGLE_API_KEY")
except (ImportError, KeyError, RuntimeError):
    secret = None
if not secret:
    secret = os.environ.get("GOOGLE_API_KEY")
if not secret:
    secret = getpass.getpass("GOOGLE_API_KEY (hidden input): ")
if not secret.strip():
    raise RuntimeError("Gemini API key required for live Operations evaluation")
os.environ["GOOGLE_API_KEY"] = secret.strip()
del secret
os.environ.setdefault("AGENTICX_MODEL", "gemini-3.5-flash-lite")
operations_agent = OperationsAgent()
print("Real Operations Agent initialized; model:", os.environ["AGENTICX_MODEL"])
print("API key is not printed or stored in notebook outputs")

Real Operations Agent initialized; model: gemini-3.5-flash-lite
API key is not printed or stored in notebook outputs


### Cell 10 — Real Operations Evaluation

In [12]:
case, options = get_case("SCR-001")
live_option = next(o for o in options if OperationsAgent.check_constraints(case, o)[0])
live_result = operations_agent.evaluate(case, live_option)
assert isinstance(live_result, AgentEvaluation)
assert live_result.agent == "Operations" and live_result.option_id == live_option["option_id"]
assert 0 <= live_result.score <= 100 and live_result.reason.strip()
assert live_result.blocking is True
print("Case:", case["case_id"], "Option:", live_option["option_id"])
display(pd.DataFrame([live_result.model_dump()]))
print("Real Gemini Operations evaluation completed")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Case: SCR-001 Option: SCR-001-O1


,agent,option_id,approved,score,reason,blocking
0,Operations,SCR-001-O1,True,100.0,Operational gates passed (duration 40 min / av...,True


Real Gemini Operations evaluation completed


### Cell 11 — ACL Ontology & Message Validation

In [13]:
bus = MessageBus(case["case_id"])
cfp = bus.send(1, "Coordinator", "ALL", "CFP", "Request operational proposals")
request = bus.send(1, "Coordinator", "Operations", "REQUEST", f"Evaluate {live_option['option_id']}")
reply = bus.send(1, "Operations", "Coordinator", "INFORM", live_result.reason, in_reply_to=request.message_id)
assert reply.in_reply_to == request.message_id
assert all(msg.case_id == case["case_id"] and msg.ontology == "studio67.event_crisis" for msg in bus.messages)
assert bus.history(agent="Operations") == [cfp, request, reply]
assert len(bus.history(agent="Operations")) == 3

def expect_error(fn):
    try:
        fn()
    except (ValueError, TypeError):
        return
    raise AssertionError("Expected invalid ACL operation to fail")

expect_error(lambda: bus.send(0, "Coordinator", "Operations", "REQUEST", "bad round"))
expect_error(lambda: bus.send(1, "", "Operations", "REQUEST", "empty sender"))
expect_error(lambda: bus.send(1, "Coordinator", "Operations", "UNKNOWN", "bad type"))
expect_error(lambda: bus.send(1, "Operations", "Coordinator", "INFORM", "bad ref", in_reply_to="missing"))
expect_error(lambda: bus.send(1, "Safety", "Coordinator", "INFORM", "wrong author", in_reply_to=request.message_id))
expect_error(lambda: bus.send(1, "Operations", "ALL", "INFORM", "wrong receiver", in_reply_to=request.message_id))
print("ACL schema, ontology, reply linkage and invalid-input tests passed")

ACL schema, ontology, reply linkage and invalid-input tests passed


### Cell 12 — Agent Communication With Real Decisions

In [14]:
finance_result = FinanceAgent().evaluate(case, live_option)
safety_result = SafetyAgent().evaluate(case, live_option)
agent_results = [live_result, finance_result, safety_result]
assert {x.agent for x in agent_results} == {"Operations", "Finance", "Safety"}

log = MessageBus(case["case_id"])
log.send(1, "Coordinator", "ALL", "CFP", "Request evaluations for " + live_option["option_id"])
log.send(1, "Coordinator", "ALL", "PROPOSE", "Candidate: " + live_option["option_id"])
for result in agent_results:
    req = log.send(1, "Coordinator", result.agent, "REQUEST", "Assess " + result.option_id)
    log.send(1, result.agent, "Coordinator", "INFORM", result.reason, in_reply_to=req.message_id)
    log.send(1, result.agent, "ALL", "ACCEPT" if result.approved else "REFUSE", result.option_id + ": " + result.reason)

# A genuine deterministic refusal from an impossible deadline.
zero_time_result = gate_agent.evaluate({**case, "time_left_min": 0}, live_option)
assert not zero_time_result.approved
log.send(1, "Operations", "ALL", "REFUSE", "Zero-time scenario test: " + zero_time_result.reason)
required_types = {"REQUEST", "INFORM", "CFP", "PROPOSE", "ACCEPT", "REFUSE"}
observed_types = {m.performative for m in log.messages}
# ACCEPT is only claimed when a real agent approves; otherwise use a separately verified real case.
if "ACCEPT" not in observed_types:
    approved_option = next(
        (o for o in options if FinanceAgent().evaluate(case, o).approved),
        None
    )
    assert approved_option is not None, "No genuine accepted finance option in case"
    actual_acceptance = FinanceAgent().evaluate(case, approved_option)
    assert actual_acceptance.approved
    log.send(1, "Finance", "ALL", "ACCEPT", actual_acceptance.option_id + ": " + actual_acceptance.reason)
assert required_types <= {m.performative for m in log.messages}
assert all(m.ontology == "studio67.event_crisis" for m in log.messages)
messages_df = pd.DataFrame(log.to_records())
display(messages_df[["message_id", "round", "sender", "receiver", "performative", "in_reply_to", "ontology", "content"]])
print("Six ACL performatives recorded from real rules and real agent decisions")

,message_id,round,sender,receiver,performative,in_reply_to,ontology,content
0,SCR-001-M0001,1,Coordinator,ALL,CFP,None,studio67.event_crisis,Request evaluations for SCR-001-O1
1,SCR-001-M0002,1,Coordinator,ALL,PROPOSE,None,studio67.event_crisis,Candidate: SCR-001-O1
2,SCR-001-M0003,1,Coordinator,Operations,REQUEST,None,studio67.event_crisis,Assess SCR-001-O1
3,SCR-001-M0004,1,Operations,Coordinator,INFORM,SCR-001-M0003,studio67.event_crisis,Operational gates passed (duration 40 min / av...
4,SCR-001-M0005,1,Operations,ALL,ACCEPT,None,studio67.event_crisis,SCR-001-O1: Operational gates passed (duration...
5,SCR-001-M0006,1,Coordinator,Finance,REQUEST,None,studio67.event_crisis,Assess SCR-001-O1
6,SCR-001-M0007,1,Finance,Coordinator,INFORM,SCR-001-M0006,studio67.event_crisis,"Committed spending 0.00 SAR + option cost 1,29..."
7,SCR-001-M0008,1,Finance,ALL,ACCEPT,None,studio67.event_crisis,SCR-001-O1: Committed spending 0.00 SAR + opti...
8,SCR-001-M0009,1,Coordinator,Safety,REQUEST,None,studio67.event_crisis,Assess SCR-001-O1
9,SCR-001-M0010,1,Safety,Coordinator,INFORM,SCR-001-M0009,studio67.event_crisis,Safety OK: True; Permit OK: True; Mitigation v...


Six ACL performatives recorded from real rules and real agent decisions


### Cell 13 — ACL Message Sequence & Multi-Round Validation

In [15]:
sequence = MessageBus("SCR-001")
a = sequence.send(1, "Coordinator", "Operations", "REQUEST", "Check resources")
b = sequence.send(1, "Operations", "Coordinator", "INFORM", "Resources evaluated", in_reply_to=a.message_id)
c = sequence.send(2, "Coordinator", "ALL", "CFP", "Request a revised proposal")
assert [m.message_id for m in sequence.messages] == ["SCR-001-M0001", "SCR-001-M0002", "SCR-001-M0003"]
assert b.in_reply_to == a.message_id and c.round == 2
expect_error(lambda: sequence.send(1, "Coordinator", "ALL", "CFP", "Backwards round"))
assert sequence.history(agent="Operations") == [a, b, c]
assert sequence.history(performative="INFORM") == [b]
print("ACL history, IDs, broadcast and round order tests passed")

ACL history, IDs, broadcast and round order tests passed


### Cell 14 — Integration Compatibility

In [16]:
from core.negotiation import NegotiationEngine
from core.runner import build_engine
assert all(hasattr(OperationsAgent, x) for x in ("name", "evaluate", "check_constraints"))
assert all(hasattr(MessageBus, x) for x in ("send", "history", "to_records"))
assert callable(NegotiationEngine) and callable(build_engine)
assert operations_agent.name == "Operations"
assert all(isinstance(x, AgentEvaluation) for x in agent_results)
assert all(isinstance(x, ACLMessage) for x in log.messages)
print("Operations and MessageBus interfaces are compatible with NegotiationEngine")

Operations and MessageBus interfaces are compatible with NegotiationEngine


### Cell 15 — Save Results

In [17]:
results_dir = ROOT / "results"
results_dir.mkdir(exist_ok=True)
resource_df.to_csv(results_dir / "operations_resource_gates.csv", index=False)
messages_df.to_csv(results_dir / "operations_acl_messages.csv", index=False)
pd.DataFrame([x.model_dump() for x in agent_results]).to_csv(
    results_dir / "operations_cross_agent_evaluations.csv", index=False
)
print("Saved 3 CSV files in:", results_dir)

Saved 3 CSV files in: /content/AgenticX/results


### Cell 16 — Conclusion

Operations combines deterministic resource constraints with a real Gemini assessment for feasible options. The communication layer records validated ACL interactions using six performatives and the shared ontology. The notebook tests three incident types, input failures, 450 resource-gate decisions, real cross-agent evaluations, and compatibility with the negotiation module. Multi-round agreements and the live UI are evaluated in Notebooks 04 and 05.